In [21]:
import wandb
import pandas as pd

ENTITY = "laithzumot"
PROJECT = "huggingface" 
RUN_ID = "yqnbvuec"

api = wandb.Api()
run = api.run(f"{ENTITY}/{PROJECT}/runs/{RUN_ID}")


In [12]:
all_steps = list(run.scan_history())
print(f"✅ Retrieved {len(all_steps)} rows from scan_history()")


✅ Retrieved 2000 rows from scan_history()


In [14]:
# Build DataFrame manually to avoid pandas issues
data = []
for step_data in all_steps:
    step = step_data.get('_step')
    reward = step_data.get('train/rewards/lean_reward')
    
    # Only add if we have both values
    if step is not None and reward is not None:
        data.append({'step': int(step), 'reward': float(reward)})
    elif step is not None:
        data.append({'step': int(step), 'reward': None})  # Explicit None for missing rewards

In [15]:
df = pd.DataFrame(data)

In [16]:
print(f"✅ Created DataFrame with {len(df)} rows")
print(f"Step range: {df['step'].min()} to {df['step'].max()}")
print(f"Non-null rewards: {df['reward'].notna().sum()}")

# Show sample before saving
print("\n=== SAMPLE DATA (first 20 rows) ===")
print(df.head(20).to_string(index=False))

# Verify data types
print(f"\nData types: {df.dtypes.to_dict()}")

# Save with explicit formatting
csv_path = '/home/lyz/repos/open-r1-lean/past_runs/run_3_algebra/step_rewards_clean.csv'
df.to_csv(csv_path, index=False, float_format='%.6f')

print(f"\n✅ SAVED TO: {csv_path}")

# Verify the file was written correctly
print("\n=== VERIFYING FILE ===")
loaded_df = pd.read_csv(csv_path)
print(f"Loaded {len(loaded_df)} rows from file")
print(loaded_df.head(10).to_string(index=False))

✅ Created DataFrame with 2000 rows
Step range: 0 to 1637
Non-null rewards: 22

=== SAMPLE DATA (first 20 rows) ===
 step  reward
    0     NaN
    1     NaN
    4     NaN
    7     NaN
    8     NaN
    9     NaN
   11     NaN
   12     NaN
   14     NaN
   16     NaN
   17     NaN
   27     NaN
   28     NaN
   30     NaN
   31     NaN
   32     NaN
   33     NaN
   36     NaN
   38     NaN
   40     NaN

Data types: {'step': dtype('int64'), 'reward': dtype('float64')}

✅ SAVED TO: /home/lyz/repos/open-r1-lean/past_runs/run_3_algebra/step_rewards_clean.csv

=== VERIFYING FILE ===
Loaded 2000 rows from file
 step  reward
    0     NaN
    1     NaN
    4     NaN
    7     NaN
    8     NaN
    9     NaN
   11     NaN
   12     NaN
   14     NaN
   16     NaN


In [ ]:
import pandas as pd
import json
import os

# --- CONFIGURE ---
# These are the ONLY steps that have individual completion data
# Download these files from wandb UI → Files → media/table/
TARGET_FILES = {
    990: "media_table_completions_990_f6175506e537dd51fe0f.table.json",
    1092: "media_table_completions_1092_1bf76a477b516fcc9dae.table.json"
}

BASE_DIR = "/home/lyz/repos/open-r1-lean/past_runs/run_3_algebra/"
# -----------------

print("=== LOADING SUCCESSFUL COMPLETIONS FROM LOCAL FILES ===")

successful_completions = []

for step, filename in TARGET_FILES.items():
    file_path = os.path.join(BASE_DIR, filename)
    
    print(f"\n--- Processing step {step} ---")
    
    if not os.path.exists(file_path):
        print(f"✗ FILE NOT FOUND: {file_path}")
        print(f"  Please download manually from:")
        print(f"  https://wandb.ai/laithzumot/huggingface/runs/yqnbvuec/files")
        print(f"  Location: media/table/{filename}")
        print(f"  Save to: {BASE_DIR}")
        continue
    
    try:
        with open(file_path, 'r') as f:
            table_data = json.load(f)
        
        df = pd.DataFrame(table_data['data'], columns=table_data['columns'])
        
        # Filter for successful completions only
        successes = df[df['reward'] == 1.0].copy()
        successes['step'] = step
        
        print(f"  ✓ Total completions: {len(df)}")
        print(f"  ✓ Successful completions: {len(successes)}")
        
        if len(successes) > 0:
            successful_completions.append(successes)
            
            # Show first successful proof
            first_success = successes.iloc[0]
            if 'completion' in first_success:
                print(f"  ✓ Sample proof (first 300 chars):")
                print(f"    {first_success['completion'][:300]}...")
        
    except Exception as e:
        print(f"  ✗ Error loading file: {e}")

print(f"\n{'='*50}")

if successful_completions:
    df_final = pd.concat(successful_completions, ignore_index=True)
    
    print(f"✅ TOTAL SUCCESSFUL PROOFS: {len(df_final)}")
    print(f"✅ FROM STEPS: {sorted(df_final['step'].unique())}")
    
    # Save full data
    csv_path = os.path.join(BASE_DIR, 'successful_completions.csv')
    df_final.to_csv(csv_path, index=False)
    print(f"✅ SAVED TO: {csv_path}")
    print(f"   File size: {os.path.getsize(csv_path):,} bytes")
    
    # Save light version (step & completion only)
    if 'completion' in df_final.columns:
        df_light = df_final[['step', 'completion']].copy()
        light_path = os.path.join(BASE_DIR, 'successful_proofs.csv')
        df_light.to_csv(light_path, index=False)
        print(f"✅ ALSO SAVED: {light_path}")
    
else:
    print("❌ No successful completions loaded")
    print("   Download the files and re-run this script")

print(f"\n{'='*50}")
print("INSTRUCTIONS:")
print("1. Go to https://wandb.ai/laithzumot/huggingface/runs/yqnbvuec/files")
print("2. Navigate to 'media/table/' folder")
print("3. Download the two completion files listed above")
print("4. Save them to:", BASE_DIR)
print("5. Re-run this script")

=== LOADING SUCCESSFUL COMPLETIONS FROM LOCAL FILES ===

--- Processing step 990 ---
✗ FILE NOT FOUND: /home/lyz/repos/open-r1-lean/past_runs/run_3_algebra/media_table_completions_990_f6175506e537dd51fe0f.table
  Please download manually from:
  https://wandb.ai/laithzumot/huggingface/runs/yqnbvuec/files
  Location: media/table/media_table_completions_990_f6175506e537dd51fe0f.table
  Save to: /home/lyz/repos/open-r1-lean/past_runs/run_3_algebra/

--- Processing step 1092 ---
  ✓ Total completions: 32
  ✓ Successful completions: 0

❌ No successful completions loaded
   Download the files and re-run this script

INSTRUCTIONS:
1. Go to https://wandb.ai/laithzumot/huggingface/runs/yqnbvuec/files
2. Navigate to 'media/table/' folder
3. Download the two completion files listed above
4. Save them to: /home/lyz/repos/open-r1-lean/past_runs/run_3_algebra/
5. Re-run this script
